# Link Graph — Bipartite & Double-Circle

Each record links a **source** document (`id`) to a **target** document (`linked_to`), including self-links.  
Two layouts are produced:

- **Layout A — Bipartite**: source nodes on the left, target nodes on the right. Self-links are horizontal.
- **Layout B — Double circle**: source nodes on the outer ring, target nodes on the inner ring at the same angle. Self-links are short radial spokes.

Edge colour is red; **opacity = `1 - recall_excl_input`** (low recall → bold edge).  
Node colour: 🟢 `correct_link = True` / 🔴 `False`.

In [ ]:
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import json

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
NAME ="af_iter_5_anon"
DATA = "MEDQA"
INPUT_FILE  = f"./final/{DATA}/{NAME}/results/per_doc_rand3.jsonl"          # <-- hardcoded path to your JSONL file
OUT_BPRT    = f"./final/{DATA}/{NAME}/link_graph_bipartite.png"
OUT_DCIR    = f"./final/{DATA}/{NAME}/link_graph_doublecircle.png"

BG          = "white"
EDGE_RGB    = (0.85, 0.1, 0.1)      # red
HIT_COLOR   = "#2ecc71"             # correct_link = True
MISS_COLOR  = "#e74c3c"             # correct_link = False
NODE_SIZE   = 5                    # scatter s=
LW          = 1.5                   # edge linewidth
DPI         = 200
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def node_colors(records):
    return [HIT_COLOR if r.get("correct_link") else MISS_COLOR for r in records]

def edge_alpha(rec):
    print(rec)
    if not rec.get("recall_excl_input"):
        return 0.04
    a = 1.0 - rec.get("recall_excl_input", 0.0)
    return max(0.0, min(1.0, a))

def legend_patches():
    return [
        mpatches.Patch(color=HIT_COLOR,           label="Correct link"),
        mpatches.Patch(color=MISS_COLOR,           label="Incorrect link"),
        mpatches.Patch(color=(*EDGE_RGB, 0.85),    label="Edge  (Opacity = Semantic Leakage)"),
    ]

records = load(INPUT_FILE)
n = len(records)
print(f"Loaded {n} records")

In [ ]:
sum(rec["id"] == rec["linked_to"] for rec in records)

## Layout A — Bipartite
Source nodes (left) → target nodes (right). Self-links are horizontal lines.

In [ ]:
def draw_bipartite(records, save_path=None):
    n = len(records)

    # Y positions: spread 0..1, node i at y = i/(n-1)
    ys = [i / (n - 1) for i in range(n)]
    src_x, tgt_x = 0.0, 1.0

    # Build edge segments
    segments, colors = [], []
    for rec in records:
        src, dst = rec["id"], rec["linked_to"]
        segments.append([(src_x, ys[src]), (tgt_x, ys[dst])])
        colors.append((*EDGE_RGB, edge_alpha(rec)))

    fig, ax = plt.subplots(figsize=(6, 10), facecolor=BG)
    ax.set_facecolor(BG)
    ax.axis("off")

    # Edges first
    lc = LineCollection(segments, colors=colors, linewidths=LW, zorder=1)
    ax.add_collection(lc)

    # Source nodes (left)
    nc = node_colors(records)
    ax.scatter([src_x] * n, ys, s=NODE_SIZE, c=nc, zorder=2, linewidths=0)

    # Target nodes (right) — colour by whether they ARE the correct target
    # We colour by the source record's correct_link for consistent meaning
    ax.scatter([tgt_x] * n, ys, s=NODE_SIZE, c=nc, zorder=2, linewidths=0)

    # Column labels
    ax.text(src_x, 1.015, "Source (id)",      ha="center", va="bottom", fontsize=10, color="#333")
    ax.text(tgt_x, 1.015, "Target (linked_to)",ha="center", va="bottom", fontsize=10, color="#333")

    ax.set_xlim(-0.15, 1.15)
    ax.set_ylim(-0.02, 1.06)
    ax.set_title(f"Bipartite link graph  |  {n} nodes  |  {len(segments)} edges",
                 fontsize=12, pad=10, color="black")
    ax.legend(handles=legend_patches(), loc="upper center",
              facecolor="white", edgecolor="#aaa", fontsize=8, ncol=3,
              bbox_to_anchor=(0.5, -0.01))

    if save_path:
        fig.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor=BG)
        print(f"Saved {save_path}")
    plt.show()

draw_bipartite(records, OUT_BPRT)

## Layout B — Double circle
Source nodes on the **outer ring**, target nodes on the **inner ring**, both at the same angle for each document id. Self-links are short radial spokes; cross-links arc between rings.

In [ ]:
def draw_double_circle(records, save_path=None):
    n = len(records)
    R_outer = 1.0
    R_inner = 0.65

    angles = [2 * math.pi * i / n - math.pi / 2 for i in range(n)]
    outer_x = [R_outer * math.cos(a) for a in angles]
    outer_y = [R_outer * math.sin(a) for a in angles]
    inner_x = [R_inner * math.cos(a) for a in angles]
    inner_y = [R_inner * math.sin(a) for a in angles]

    # Build edge segments: outer[src] -> inner[dst]
    segments, colors = [], []
    for rec in records:
        src, dst = rec["id"], rec["linked_to"]
        segments.append([
            (outer_x[src], outer_y[src]),
            (inner_x[dst], inner_y[dst])
        ])
        colors.append((*EDGE_RGB, edge_alpha(rec)))

    fig, ax = plt.subplots(figsize=(5, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.set_aspect("equal")
    ax.axis("off")

    # Edges
    lc = LineCollection(segments, colors=colors, linewidths=LW, zorder=1)
    ax.add_collection(lc)

    # Outer nodes (sources)
    nc = node_colors(records)
    ax.scatter(outer_x, outer_y, s=NODE_SIZE, c=nc, zorder=2, linewidths=0)

    # Inner nodes (targets) — same colour scheme
    ax.scatter(inner_x, inner_y, s=NODE_SIZE, c=nc, zorder=2, linewidths=0)

    # Ring labels
    ax.text(0, R_outer + 0.07,  "Outer: Source (id)",       ha="center", fontsize=10, color="#333")
    ax.text(0, R_inner + 0.07,  "Inner: Target (linked_to)", ha="center", fontsize=10, color="#333")

    margin = 0.2
    ax.set_xlim(-R_outer - margin, R_outer + margin)
    ax.set_ylim(-R_outer - margin, R_outer + margin)
    ax.set_title(f"Double-circle link graph  |  {n} nodes  |  {len(segments)} edges",
                 fontsize=13, pad=12, color="black")
    ax.legend(handles=legend_patches(), loc="lower center",
              facecolor="white", edgecolor="#aaa", fontsize=9)

    if save_path:
        fig.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor=BG)
        print(f"Saved {save_path}")
    plt.show()

draw_double_circle(records, OUT_DCIR)